# Day 11 v1 — Gold: Realtime Charging Sessions Enriched (Blob Silver → Gold)

**Learning version — one Gold mart, every join step explicit.**

**Source (Silver):**
- `silver-volume/realtime/charging_sessions/` — realtime IoT CSV data (Day 9)
- `silver-volume/api/stations/`               — station dimension (Day 8)
- `silver-volume/api/chargers/`               — charger dimension (Day 8)
- `silver-volume/api/customers/`              — customer dimension (Day 8)
- `silver-volume/api/tariffs/`                — tariff/pricing dimension (Day 8)
- `silver-volume/api/energy_prices/`          — regional energy price (Day 8)

**Sink:** `gold-volume/blob/realtime_sessions_enriched/` (Delta, full overwrite)

**What this mart answers:**
- Realtime session data enriched with station, charger, customer context and pricing
- Device telemetry metrics (temp, signal, firmware) alongside business metrics
- Energy cost vs tariff price comparison per session

**Steps:**
1. Read Silver realtime sessions + API dimension tables
2. Join sessions with dimensions
3. Compute derived metrics (estimated_cost, energy_rate_comparison)
4. Add Gold audit columns
5. Write Gold Delta (full overwrite)
6. Verify output

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
SILVER_REALTIME = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
SILVER_API      = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD_BLOB       = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/blob'
PIPELINE        = 'pl_gold_blob_realtime_enriched_v1'
RUN_TS          = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Silver realtime : {SILVER_REALTIME}')
print(f'Silver API      : {SILVER_API}')
print(f'Gold blob       : {GOLD_BLOB}')
print(f'RUN_TS          : {RUN_TS}')

In [ ]:
# Cell 3 — Read Silver tables
rt_sessions  = spark.read.format('delta').load(f'{SILVER_REALTIME}/charging_sessions')
stations     = spark.read.format('delta').load(f'{SILVER_API}/stations')
chargers     = spark.read.format('delta').load(f'{SILVER_API}/chargers')
customers    = spark.read.format('delta').load(f'{SILVER_API}/customers')
tariffs      = spark.read.format('delta').load(f'{SILVER_API}/tariffs')
energy_prices= spark.read.format('delta').load(f'{SILVER_API}/energy_prices')

print(f'realtime sessions : {rt_sessions.count():>8} rows')
print(f'stations          : {stations.count():>8} rows')
print(f'chargers          : {chargers.count():>8} rows')
print(f'customers         : {customers.count():>8} rows')
print(f'tariffs           : {tariffs.count():>8} rows')
print(f'energy_prices     : {energy_prices.count():>8} rows')

In [ ]:
# Cell 4 — Prepare dimension tables
sta_dim = stations.select(
    'station_id',
    F.col('name').alias('station_name'),
    F.col('city').alias('station_city'),
    F.col('state').alias('station_state'),
    F.col('country').alias('station_country'),
    F.col('latitude').alias('station_lat'),
    F.col('longitude').alias('station_lon'),
)

chg_dim = chargers.select(
    'charger_id',
    'charger_type',
    F.col('power_kw').alias('charger_power_kw'),
    F.col('status').alias('charger_status'),
)

cust_dim = customers.select(
    'customer_id',
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
)

tar_dim = tariffs.select(
    'tariff_id',
    F.col('name').alias('tariff_name'),
    F.col('price_per_kwh').alias('tariff_price_per_kwh'),
    F.col('price_per_min').alias('tariff_price_per_min'),
)

# Latest energy price per region (some regions have multiple effective periods)
from pyspark.sql.window import Window
ep_win = Window.partitionBy('region').orderBy(F.col('effective_from').desc())
ep_dim = (
    energy_prices
    .withColumn('_rn', F.row_number().over(ep_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('region').alias('station_state'),   # join on state
        F.col('price_per_kwh').alias('grid_price_per_kwh'),
        F.col('currency').alias('grid_currency'),
    )
)

print('Dimension tables prepared')

In [ ]:
# Cell 5 — Join and compute derived metrics
gold_df = (
    rt_sessions
    .join(sta_dim,  on='station_id',  how='left')
    .join(chg_dim,  on='charger_id',  how='left')
    .join(cust_dim, on='customer_id', how='left')
    .join(tar_dim,  on='tariff_id',   how='left')
    .join(ep_dim,   on='station_state', how='left')
    # Date parts for partitioning
    .withColumn('session_date',  F.to_date('plug_in_ts'))
    .withColumn('session_year',  F.year('plug_in_ts'))
    .withColumn('session_month', F.month('plug_in_ts'))
    .withColumn('session_hour',  F.hour('plug_in_ts'))
    # Estimated cost from tariff
    .withColumn(
        'estimated_cost_aud',
        F.when(
            F.col('tariff_price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('tariff_price_per_kwh') * F.col('energy_kwh')
             + F.col('tariff_price_per_min') * F.col('duration_min')).cast('decimal(10,2)')
        )
    )
    # Grid cost (what the station pays for the energy)
    .withColumn(
        'grid_cost_aud',
        F.when(
            F.col('grid_price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('grid_price_per_kwh') * F.col('energy_kwh')).cast('decimal(10,2)')
        )
    )
    # Gross margin per session (revenue - grid cost)
    .withColumn(
        'gross_margin_aud',
        F.when(
            F.col('estimated_cost_aud').isNotNull() & F.col('grid_cost_aud').isNotNull(),
            (F.col('estimated_cost_aud') - F.col('grid_cost_aud')).cast('decimal(10,2)')
        )
    )
    # Charger utilisation during session (session duration as % of charger rated power delivered)
    .withColumn(
        'power_efficiency_pct',
        F.when(
            F.col('charger_power_kw').isNotNull() & (F.col('charger_power_kw') > 0)
            & F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
            (F.col('energy_kwh') / (F.col('charger_power_kw') * F.col('duration_min') / 60) * 100
            ).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'Gold rows : {gold_df.count()}')
print(f'Gold cols : {len(gold_df.columns)}')
gold_df.printSchema()

In [ ]:
# Cell 6 — Write Gold Delta (full overwrite)
gold_path = f'{GOLD_BLOB}/realtime_sessions_enriched'

(
    gold_df.write.format('delta')
    .mode('overwrite').option('overwriteSchema', 'true')
    .partitionBy('session_year', 'session_month')
    .save(gold_path)
)

print(f'Written to : {gold_path}')
print(f'Rows       : {gold_df.count()}')

In [ ]:
# Cell 7 — Verify Gold output
g = spark.read.format('delta').load(gold_path)
print(f'Total Gold rows : {g.count()}')
g.show(3, truncate=True)

print('\nAvg power efficiency by charger type:')
g.groupBy('charger_type').agg(
    F.count('session_id').alias('sessions'),
    F.round(F.avg('power_efficiency_pct'), 2).alias('avg_efficiency_pct'),
    F.round(F.avg('energy_kwh'), 4).alias('avg_energy_kwh'),
).orderBy('charger_type').show(truncate=False)

print('\nNull check on session_id (should be 0):')
print(g.filter(F.col('session_id').isNull()).count())